# Anomaly Detection on QLD Government Open Data

**Data Source:** TMR New Business Registration Transactions (data.qld.gov.au)  
**Records:** 304,553 transactions across 654 suburbs, 12 months  
**Snowflake Features:** `SNOWFLAKE.ML.ANOMALY_DETECTION`, `CORTEX.COMPLETE`, Notebooks

In [ ]:
-- Set notebook context
USE DATABASE CDSB_DEMO;
USE SCHEMA RAW;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
!pip install plotly neo4j-viz --quiet

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print('Connected to', session.get_current_database(), session.get_current_schema())

## 0. Load Data from Open Data Portal
The TMR CSV is already staged at `@QLD_OPEN_DATA/tmr/` - load it into tables and create aggregations

In [ ]:
CREATE STAGE IF NOT EXISTS QLD_OPEN_DATA;

In [ ]:
CREATE FILE FORMAT IF NOT EXISTS CSV_FORMAT
    SKIP_HEADER = 1
    DATE_FORMAT = 'DD-MM-YYYY'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"';

In [ ]:
CREATE OR REPLACE TABLE TMR_REGISTRATIONS (
    MONTH DATE,
    SUBURB VARCHAR,
    POSTCODE VARCHAR,
    TRANSACTION_TYPE VARCHAR,
    MAKE_NAME VARCHAR,
    MODEL_NAME VARCHAR,
    VALID_HANDLED_TRANSACTIONS NUMBER
);

COPY INTO TMR_REGISTRATIONS
FROM @QLD_OPEN_DATA/tmr/
FILE_FORMAT = CSV_FORMAT
ON_ERROR = 'CONTINUE';

In [ ]:
CREATE OR REPLACE TABLE TMR_SUBURB_MONTHLY AS
SELECT DATE_TRUNC('MONTH', MONTH) as TS,
       SUBURB as SERIES_ID,
       SUM(VALID_HANDLED_TRANSACTIONS) as REGISTRATIONS
FROM TMR_REGISTRATIONS
GROUP BY DATE_TRUNC('MONTH', MONTH), SUBURB;

In [ ]:
CREATE OR REPLACE VIEW TMR_MONTHLY_REGISTRATIONS AS
SELECT MONTH,
       SUM(VALID_HANDLED_TRANSACTIONS) as TOTAL_REGISTRATIONS,
       COUNT(DISTINCT SUBURB) as ACTIVE_SUBURBS,
       COUNT(DISTINCT MAKE_NAME) as UNIQUE_MAKES
FROM TMR_REGISTRATIONS
GROUP BY MONTH ORDER BY MONTH;

In [ ]:
CREATE OR REPLACE VIEW TMR_REGIONAL_REGISTRATIONS AS
SELECT MONTH,
       CASE
           WHEN LEFT(POSTCODE, 2) = '40' THEN 'Brisbane South'
           WHEN LEFT(POSTCODE, 2) = '41' THEN 'Brisbane South/Logan'
           WHEN LEFT(POSTCODE, 2) = '42' THEN 'Gold Coast/Scenic Rim'
           WHEN LEFT(POSTCODE, 2) = '43' THEN 'Ipswich/Lockyer'
           WHEN LEFT(POSTCODE, 2) = '44' THEN 'Toowoomba/Darling Downs'
           WHEN LEFT(POSTCODE, 2) = '45' THEN 'Brisbane North'
           WHEN LEFT(POSTCODE, 2) = '46' THEN 'Sunshine Coast/Wide Bay'
           WHEN LEFT(POSTCODE, 2) = '47' THEN 'Bundaberg/Gladstone'
           WHEN LEFT(POSTCODE, 2) = '48' THEN 'Rockhampton/Mackay'
           WHEN LEFT(POSTCODE, 2) = '49' THEN 'Townsville/Far North'
           ELSE 'Other'
       END as REGION,
       SUM(VALID_HANDLED_TRANSACTIONS) as TOTAL_REGISTRATIONS,
       COUNT(DISTINCT SUBURB) as ACTIVE_SUBURBS
FROM TMR_REGISTRATIONS
GROUP BY MONTH, REGION ORDER BY MONTH, REGION;

## 1. Explore the Data
Real QLD Government open data - 304K vehicle registration transactions from TMR

In [ ]:
SELECT MONTH, SUBURB, POSTCODE, TRANSACTION_TYPE, MAKE_NAME, MODEL_NAME,
       VALID_HANDLED_TRANSACTIONS
FROM TMR_REGISTRATIONS
LIMIT 10

In [ ]:
df_monthly = session.sql('SELECT * FROM TMR_MONTHLY_REGISTRATIONS').to_pandas()

fig = px.bar(df_monthly, x='MONTH', y='TOTAL_REGISTRATIONS',
             title='QLD Monthly New Business Vehicle Registrations',
             labels={'TOTAL_REGISTRATIONS': 'Registrations', 'MONTH': 'Month'})
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
df_regional = session.sql('SELECT * FROM TMR_REGIONAL_REGISTRATIONS').to_pandas()

fig = px.line(df_regional, x='MONTH', y='TOTAL_REGISTRATIONS', color='REGION',
              title='Registrations by QLD Region',
              labels={'TOTAL_REGISTRATIONS': 'Registrations', 'MONTH': 'Month'})
fig.update_layout(template='plotly_white')
fig.show()

## 2. Train Anomaly Detection Model

One SQL statement - Snowflake handles feature engineering, model selection, and training.  
We use **multi-series** mode: train one model across all 654 suburbs simultaneously.

In [ ]:
CREATE OR REPLACE VIEW TMR_TRAINING_DATA AS
SELECT TS, SERIES_ID, REGISTRATIONS
FROM TMR_SUBURB_MONTHLY
WHERE TS < '2026-02-01'
  AND SERIES_ID IN (
    SELECT SERIES_ID FROM TMR_SUBURB_MONTHLY
    WHERE TS < '2026-02-01'
    GROUP BY SERIES_ID HAVING COUNT(DISTINCT TS) >= 8
  )

In [ ]:
CREATE OR REPLACE VIEW TMR_TEST_DATA AS
SELECT TS, SERIES_ID, REGISTRATIONS
FROM TMR_SUBURB_MONTHLY
WHERE TS >= '2026-02-01'
  AND SERIES_ID IN (
    SELECT SERIES_ID FROM TMR_SUBURB_MONTHLY
    WHERE TS < '2026-02-01'
    GROUP BY SERIES_ID HAVING COUNT(DISTINCT TS) >= 8
  )

In [ ]:
train_count = session.sql('SELECT COUNT(*) as N FROM TMR_TRAINING_DATA').to_pandas().iloc[0,0]
test_count = session.sql('SELECT COUNT(*) as N FROM TMR_TEST_DATA').to_pandas().iloc[0,0]
print(f'Training: {train_count} rows (Apr 2025 - Jan 2026)')
print(f'Test: {test_count} rows (Feb - Mar 2026)')

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION tmr_anomaly_model(
    INPUT_DATA => TABLE(TMR_TRAINING_DATA),
    SERIES_COLNAME => 'SERIES_ID',
    TIMESTAMP_COLNAME => 'TS',
    TARGET_COLNAME => 'REGISTRATIONS',
    LABEL_COLNAME => '',
    CONFIG_OBJECT => {'on_error': 'skip'}
)

## 3. Detect Anomalies
Run the model on recent data to find unusual registration patterns

In [ ]:
CREATE OR REPLACE TABLE TMR_ANOMALY_RESULTS AS
SELECT *
FROM TABLE(
    tmr_anomaly_model!DETECT_ANOMALIES(
        INPUT_DATA => TABLE(TMR_TEST_DATA),
        SERIES_COLNAME => 'SERIES_ID',
        TIMESTAMP_COLNAME => 'TS',
        TARGET_COLNAME => 'REGISTRATIONS',
        CONFIG_OBJECT => {'prediction_interval': 0.95}
    )
)

In [ ]:
df_anomalies = session.sql("""
    SELECT * FROM TMR_ANOMALY_RESULTS
    ORDER BY IS_ANOMALY DESC, PERCENTILE DESC
""").to_pandas()

n_anomalies = df_anomalies['IS_ANOMALY'].sum()
print(f'Detected {n_anomalies} anomalies out of {len(df_anomalies)} suburb-months')
df_anomalies.head(20)

In [ ]:
df_anom_only = df_anomalies[df_anomalies['IS_ANOMALY'] == True].copy()
df_anom_only = df_anom_only.sort_values('PERCENTILE', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_anom_only['SERIES'].head(25),
    y=df_anom_only['Y'].head(25),
    name='Actual', marker_color='red'
))
fig.add_trace(go.Bar(
    x=df_anom_only['SERIES'].head(25),
    y=df_anom_only['FORECAST'].head(25),
    name='Expected', marker_color='lightblue'
))
fig.update_layout(
    title='Top 25 Anomalous Suburbs: Actual vs Expected Registrations',
    barmode='group', template='plotly_white', xaxis_tickangle=-45
)
fig.show()

## 4. Time Series Deep Dive - Single Suburb
Pick the most anomalous suburb and visualise its full history with detection bounds

In [ ]:
target_suburb = df_anom_only.iloc[0]['SERIES'] if len(df_anom_only) > 0 else 'BRISBANE CITY'

df_suburb = session.sql(f"""
    SELECT s.TS, s.REGISTRATIONS, a.FORECAST, a.LOWER_BOUND, a.UPPER_BOUND, a.IS_ANOMALY
    FROM TMR_SUBURB_MONTHLY s
    LEFT JOIN TMR_ANOMALY_RESULTS a ON s.TS = a.TS AND s.SERIES_ID = a.SERIES
    WHERE s.SERIES_ID = '{target_suburb}'
    ORDER BY s.TS
""").to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['REGISTRATIONS'],
                         mode='lines+markers', name='Actual', line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['FORECAST'],
                         mode='lines', name='Forecast', line=dict(color='green', dash='dash')))
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['UPPER_BOUND'],
                         mode='lines', name='Upper Bound', line=dict(color='grey', dash='dot')))
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['LOWER_BOUND'],
                         mode='lines', name='Lower Bound', line=dict(color='grey', dash='dot'),
                         fill='tonexty', fillcolor='rgba(200,200,200,0.2)'))

anomaly_pts = df_suburb[df_suburb['IS_ANOMALY'] == True]
if len(anomaly_pts) > 0:
    fig.add_trace(go.Scatter(x=anomaly_pts['TS'], y=anomaly_pts['REGISTRATIONS'],
                             mode='markers', name='ANOMALY',
                             marker=dict(color='red', size=14, symbol='x')))

fig.update_layout(title=f'Registration Anomalies - {target_suburb}', template='plotly_white')
fig.show()

## 5. LLM-Powered Anomaly Interpretation
Use `CORTEX.COMPLETE` to have an AI explain the anomalies in plain English

In [ ]:
SELECT SERIES as SUBURB, TS as MONTH, Y as ACTUAL,
       ROUND(FORECAST, 0) as EXPECTED,
       ROUND(Y - FORECAST, 0) as DEVIATION,
       ROUND(PERCENTILE, 4) as CONFIDENCE
FROM TMR_ANOMALY_RESULTS
WHERE IS_ANOMALY = TRUE
ORDER BY ABS(Y - FORECAST) DESC
LIMIT 15

In [ ]:
df_top = session.sql("""
    SELECT SERIES as SUBURB, TS as MONTH, Y as ACTUAL,
           ROUND(FORECAST, 0) as EXPECTED,
           ROUND(Y - FORECAST, 0) as DEVIATION,
           ROUND(PERCENTILE, 4) as CONFIDENCE
    FROM TMR_ANOMALY_RESULTS
    WHERE IS_ANOMALY = TRUE
    ORDER BY ABS(Y - FORECAST) DESC
    LIMIT 15
""").to_pandas()

anomaly_text = df_top.to_string(index=False)

llm_result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-6',
        $$You are a government data analyst for Queensland Transport and Main Roads.
Below are detected anomalies in new business vehicle registration transactions across QLD suburbs.

ANOMALY DATA:
{anomaly_text}

Please:
1. Summarise the top patterns (which suburbs, direction, significance)
2. Suggest possible real-world explanations
3. Recommend which anomalies need urgent investigation
4. Suggest additional data sources that would help confirm these patterns$$
    ) as ANALYSIS
""").to_pandas()

print(llm_result.iloc[0]['ANALYSIS'])

## 6. Automate with Tasks & Alerts
In production: retrain monthly, alert on new anomalies - all SQL-native

In [ ]:
-- PRODUCTION AUTOMATION (reference only)
-- Retrain model on 1st of each month:
-- CREATE OR REPLACE TASK tmr_anomaly_retrain WAREHOUSE=COMPUTE_WH
--   SCHEDULE='USING CRON 0 6 1 * * Australia/Brisbane'
-- AS CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION tmr_anomaly_model(...);

-- Weekly alert for new anomalies:
-- CREATE OR REPLACE ALERT tmr_anomaly_alert ...

SELECT 'See comments above for production Task + Alert automation' as NOTE

## Key Takeaways

| Step | Snowflake Feature | Effort |
|------|------------------|--------|
| Load open data | Stage + COPY INTO | 2 min |
| Train model | `SNOWFLAKE.ML.ANOMALY_DETECTION` | 1 SQL statement |
| Detect anomalies | `model!DETECT_ANOMALIES()` | 1 SQL statement |
| Explain with AI | `CORTEX.COMPLETE` | 1 SQL statement |
| Automate | Tasks + Alerts | Standard Snowflake |

**Total: from raw data to AI-powered anomaly alerts in under 30 minutes**